<a href="https://colab.research.google.com/github/Tejaswimadastu/Deep_Learning/blob/main/Encoder_Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Input,LSTM,Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [33]:
# Sample dataset
data = [
    ("i am learning", "je suis en train d apprendre"),
    ("he is running", "il court"),
    ("she is happy", "elle est heureuse"),
    ("i am happy", "je suis heureux")
]
input_text=[x[0] for x in data]
target_text=["<start> "+x[1]+" <end>" for x in data] # Added spaces around <start> and <end>

Tokenization

In [34]:
#Encoder Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer

input_tokenizer=Tokenizer()
input_tokenizer.fit_on_texts(input_text)
input_sequences=input_tokenizer.texts_to_sequences(input_text)

target_tokenizer=Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_text)
target_sequences=target_tokenizer.texts_to_sequences(target_text)


Padding

In [35]:
# Sample dataset
data = [
    ("i am learning", "je suis en train d apprendre"),
    ("he is running", "il court"),
    ("she is happy", "elle est heureuse"),
    ("i am happy", "je suis heureux")
]
input_text=[x[0] for x in data]
target_text=["<start> "+x[1]+" <end>" for x in data] # Corrected: Added spaces around <start> and <end>

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

input_tokenizer=Tokenizer()
input_tokenizer.fit_on_texts(input_text)
input_sequences=input_tokenizer.texts_to_sequences(input_text)

# Ensure the tokenizer learns the start and end tokens explicitly and correctly
target_tokenizer=Tokenizer(filters='', lower=True) # Keep filters='' to preserve < and >

# Collect all unique words for fitting the target tokenizer
all_target_words_for_fitting = set()
all_target_words_for_fitting.add('<start>')
all_target_words_for_fitting.add('<end>')

for sentence in target_text:
    for word in sentence.split(' '): # Split by space to get individual tokens
        if word: # Ensure no empty strings are added
            all_target_words_for_fitting.add(word)

target_tokenizer.fit_on_texts(list(all_target_words_for_fitting)) # Fit on the comprehensive list of unique words

target_sequences=target_tokenizer.texts_to_sequences(target_text)

max_input_len = max(len(seq) for seq in input_sequences)
max_target_len = max(len(seq) for seq in target_sequences)

encoder_input_data = pad_sequences(
    input_sequences,
    maxlen=max_input_len,
    padding='post'
)

decoder_input_data = pad_sequences(
    target_sequences,
    maxlen=max_target_len,
    padding='post'
)

# Update vocab_size_target based on the final word_index length
vocab_size_target = len(target_tokenizer.word_index) + 1 # +1 for 0-padding

#Create Decoder Target (Shifted)

👉 Important teaching point:

Decoder input = ...

Decoder output = ...

In [36]:
import numpy as np
decoder_target_data = np.zeros_like(decoder_input_data)

for i in range(len(data)):
    decoder_target_data[i, :-1] = decoder_input_data[i, 1:]

Build Model

parameters

In [37]:
vocab_size_input = len(input_tokenizer.word_index) + 1
vocab_size_target = len(target_tokenizer.word_index) + 1

embedding_dim = 50
latent_dim = 100

# Encoder

In [38]:
from tensorflow.keras.layers import Embedding # Added import for Embedding

encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(vocab_size_input,
                    embedding_dim)(encoder_inputs)
encoder_lstm=LSTM(latent_dim,return_state=True)
encoder_outputs,state_h,state_c=encoder_lstm(enc_emb)
encoder_states=[state_h,state_c]
print(encoder_states)

[<KerasTensor shape=(None, 100), dtype=float32, sparse=False, ragged=False, name=keras_tensor_31>, <KerasTensor shape=(None, 100), dtype=float32, sparse=False, ragged=False, name=keras_tensor_32>]


#Decoder

In [39]:
decoder_inputs = Input(shape=(None,))

dec_emb_layer = Embedding(
    vocab_size_target,
    embedding_dim
)

dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    dec_emb,
    initial_state=encoder_states
)

decoder_dense = Dense(
    vocab_size_target,
    activation='softmax'
)

decoder_outputs = decoder_dense(decoder_outputs)

print(decoder_outputs)

<KerasTensor shape=(None, None, 15), dtype=float32, sparse=False, ragged=False, name=keras_tensor_38>


# Compile model

In [40]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

In [41]:
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=1,
    epochs=200
)

Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 2.6997
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.6552
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 2.5988
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.5227
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.3374
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 1.9750
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 1.8273
Epoch 8/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 1.7369
Epoch 9/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 1.6759
Epoch 10/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 1.5899
Epoch 11/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 1.5625
Epoch 12/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 1.5470
Epoch 13/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 1.5221
Epoch 14/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 1.4945
Epoch 15/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 1.4722
Epoch 16/200
4/4 ━━

# Inference Model

Training model not equal prediction model

Encoder Interface

In [42]:
encoder_model=Model(encoder_inputs,encoder_states)

# Decoder Interface

In [43]:
decoder_state_h = Input(shape=(latent_dim,))
decoder_state_c = Input(shape=(latent_dim,))

decoder_states_inputs = [decoder_state_h, decoder_state_c]

# Define a new Input tensor for the decoder's sequence input in the inference model
decoder_sequence_input = Input(shape=(None,))

# Reuse the existing Embedding layer and apply it to the new decoder_sequence_input
dec_emb2 = dec_emb_layer(decoder_sequence_input)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2,
    initial_state=decoder_states_inputs
)

decoder_states2 = [state_h2, state_c2]

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_sequence_input] + decoder_states_inputs, # Use the new decoder_sequence_input here
    [decoder_outputs2] + decoder_states2
)

In [44]:
reverse_target_index = {i: word for word, i in target_tokenizer.word_index.items()}

def decode_sequence(input_seq):

    states_value = encoder_model.predict(input_seq)

    start_token = target_tokenizer.word_index.get('<start>')

    if start_token is None:
        start_token = target_tokenizer.word_index.get('start')

    end_token = target_tokenizer.word_index.get('<end>')

    if end_token is None:
        end_token = target_tokenizer.word_index.get('end')

    target_seq = np.array([[start_token]])

    stop_condition = False
    decoded_sentence = ""

    while not stop_condition:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        sampled_word = reverse_target_index.get(
            sampled_token_index, ''
        )

        if (
            sampled_token_index == end_token or
            len(decoded_sentence.split()) > max_target_len
        ):
            stop_condition = True
        else:
            decoded_sentence += " " + sampled_word

        target_seq = np.array([[sampled_token_index]])

        states_value = [h, c]

    return decoded_sentence

# Test

In [45]:
test_input = "i am happy"

seq = input_tokenizer.texts_to_sequences([test_input])
seq = pad_sequences(seq, maxlen=max_input_len, padding='post')

print("Input :", test_input)
print("Output:", decode_sequence(seq))

Input : i am happy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Output:  je suis heureux
